In [ ]:
# Cell 1. Kaggle P100 compatible setup

# P100 = Pascal / sm_60
# Kaggle 기본 PyTorch가 sm_60을 지원하지 않는 경우가 있으므로 CUDA 12.6 빌드로 재설치
!pip uninstall -y torch torchvision torchaudio -q
!pip install -q torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126

!pip install -q h5py onnx onnxruntime opencv-python

import os
import sys
import glob
import shlex
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/InHyunseo/AUE4040-team-project.git"
BRANCH = "avoid-oversample"   # 이 노트북이 clone 할 브랜치. 안정 버전 쓰려면 "main".
REPO_DIR = Path("/kaggle/working/AUE4040-team-project")
TRAIN_DIR = REPO_DIR / "final_project" / "training"

if not REPO_DIR.exists():
    print(f"Cloning repository (branch={BRANCH})...")
    !git clone -q -b $BRANCH $REPO $REPO_DIR
else:
    print("Repository already exists:", REPO_DIR)

%cd /kaggle/working/AUE4040-team-project/final_project/training

import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    print("capability:", torch.cuda.get_device_capability(0))

In [ ]:
import h5py
import glob
import shutil
from pathlib import Path

print("Kaggle input folders:")
!ls -lah /kaggle/input
print("\nFound h5 files:")
!find /kaggle/input -name "*.h5" | head -50

INPUT_H5 = sorted(set(glob.glob("/kaggle/input/**/*.h5", recursive=True)))
print(f"\n/kaggle/input에서 찾은 h5 개수: {len(INPUT_H5)}")

if len(INPUT_H5) == 0:
    raise FileNotFoundError(
        "h5 파일을 찾지 못함. Kaggle Notebook 우측 Add data에서 h5 파일이 들어있는 Dataset을 먼저 연결해야 함."
    )

# 맨 앞 파일들 제거

INPUT_H5 = INPUT_H5[4:]
print(f"제거 후 h5 개수: {len(INPUT_H5)}")

USE_WORKING_COPY = False
WORK_H5_DIR = Path("/kaggle/working/labels_all")
WORK_H5_DIR.mkdir(parents=True, exist_ok=True)

if USE_WORKING_COPY:
    CACHES = []
    for i, src in enumerate(INPUT_H5):
        src = Path(src)
        dst = WORK_H5_DIR / f"{i:03d}_{src.name}"
        if not dst.exists():
            print("copy:", src, "->", dst)
            shutil.copy2(src, dst)
        CACHES.append(str(dst))
else:
    CACHES = INPUT_H5

print(f"\n학습에 사용할 h5 개수: {len(CACHES)}")
for c in CACHES:
    try:
        with h5py.File(c, "r") as f:
            print(f"{c} → samples: {f['lane'].shape[0]} | keys: {list(f.keys())}")
    except Exception as e:
        print(f"{c} 파일 오류: {e}")

In [ ]:
import numpy as np
import h5py
import cv2
import matplotlib.pyplot as plt

from dataset import composite_lane, composite_front
from viz import draw_intent, GT_COLOR

# H5 raw 에서 직접 합성 + waypoint(GT) 오버레이. 정규화 텐서를 역변환하지 않으므로
# GT 좌표가 픽셀에 정확히 찍힌다. draw_intent 좌표 변환은 extract_labels 부호
# 정규화(전진=+x, 좌회전=+y)와 일치 → 직진=위, 좌코너=좌상, 우코너=우상이면 정상.
#
# WP_FIX_SIGN: 옛(부호 버그) H5 면 학습(Cell 3)과 동일하게 여기서도 wp x 만 반전해
# 봐야 시각화가 맞는다. 안 하면 점이 화면 아래로 깔려 잘려 보인다. 학습 셀의
# WP_FIX_SIGN 과 같은 값으로 둘 것.
WP_FIX_SIGN = True

_files = [h5py.File(c, "r") for c in CACHES]
_lens = [f["lane"].shape[0] for f in _files]
_cum = np.cumsum([0] + _lens)

def _get(gidx):
    fi = int(np.searchsorted(_cum, gidx, side="right") - 1)
    li = gidx - int(_cum[fi])
    f = _files[fi]
    wp = f["waypoint"][li].astype(np.float32)
    if WP_FIX_SIGN:
        wp = wp.copy(); wp[:, 0] *= -1.0   # 옛 부호 보정: x(전방)만 반전
    return dict(lane=f["lane"][li], front=f["front"][li], seg=f["seg"][li],
                det=f["det"][li], wp=wp,
                steer=float(f["steer"][li]), thr=float(f["throttle"][li]))

all_steer = np.concatenate([f["steer"][:] for f in _files])
print("total samples:", len(all_steer))

# 직진/완만한 좌·우 코너 대표 샘플
def _pick(mask):
    idx = np.where(mask)[0]
    return int(idx[len(idx) // 2]) if len(idx) else None

picks = []
for tag, m in [("straight", np.abs(all_steer) < 0.05),
               ("left turn", (all_steer < -0.3) & (all_steer > -0.6)),
               ("right turn", (all_steer > 0.3) & (all_steer < 0.6))]:
    p = _pick(m)
    if p is not None:
        picks.append((tag, p))

fig, axes = plt.subplots(len(picks), 2, figsize=(9, 3.2 * len(picks)))
if len(picks) == 1:
    axes = axes[None, :]
for row, (tag, idx) in enumerate(picks):
    s = _get(idx)
    lane_vis = draw_intent(composite_lane(s["lane"], s["seg"]), s["wp"], color=GT_COLOR)
    front_vis = composite_front(s["front"], s["det"])
    axes[row, 0].imshow(cv2.cvtColor(lane_vis, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(
        f"{tag}  steer(w)={s['steer']:+.2f}  wp[-1]=({s['wp'][-1,0]:+.2f},{s['wp'][-1,1]:+.2f})",
        fontsize=9)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(cv2.cvtColor(front_vis, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"front  throttle(v)={s['thr']:+.2f}", fontsize=9)
    axes[row, 1].axis("off")

plt.suptitle(f"waypoint GT (white) WP_FIX_SIGN={WP_FIX_SIGN} — 직진↑/좌↖/우↗ 면 정상", fontsize=10)
plt.tight_layout()
plt.show()

for f in _files:
    f.close()

In [ ]:
import os
import sys
import shlex

CKPT = "/kaggle/working/e2e_best.pt"
VIZ_DIR = "/kaggle/working/train_viz"

# ───── 미세조정(finetune) 설정 ─────────────────────────────────────────
# 기존 모델(병렬 구조, pure pursuit 시절 weaving 잡힌 best.pt)을 새 데이터로 미세조정.
#  - RESUME: 그 best.pt 경로. Kaggle 이면 /kaggle/input/<dataset>/e2e_best.pt 로 업로드.
#  - FINETUNE=True: 가중치만 불러오고 optimizer/scheduler/best 는 새로 시작 → 새 낮은 LR
#    로 깨끗이, best 도 새 데이터 기준으로 다시 잡아 조기종료 방지.
#  - 미세조정 LR 은 낮게(처음부터 학습의 1/5~1/10). 여기선 5e-5.
# 처음부터 학습하려면 RESUME="" + FINETUNE=False + LR=3e-4.
RESUME = ""              # 예: "/kaggle/input/prev-model/e2e_best.pt"
FINETUNE = False         # RESUME 채우고 미세조정할 거면 True
LR = 5e-5 if FINETUNE else 3e-4

EPOCHS = 60
BATCH = 32
VAL_FRAC = 0.15
WORKERS = 4
PATIENCE = 10         # LR plateau 로 낮춘 뒤 더 내려가는지 볼 여유 (lr_patience=3 보다 큼)

# 과적합 억제: dropout 0.5(model.py), weight_decay 1e-3(train_e2e 기본값) 코드 반영.
# split 은 세션(H5) 단위(make_splits). LR 스케줄: ReduceLROnPlateau.

# ───── loss 가중치 + 진단 손잡이 ───────────────────────────────────────
STEER_WEIGHT = 1.0
THROTTLE_WEIGHT = 0.5
WAYPOINT_WEIGHT = 0.5     # 부호 미보정 옛 H5 면 0.0 으로 끄는 게 나음
STEER_SMOOTH = 0          # 0=끔, 예: 3 (세션 내부 ±k 이동평균)

# 옛(부호 버그) H5 면 True → waypoint x 만 반전해 올바른 부호로 학습/시각화.
WP_FIX_SIGN = True

# 좌우 flip aug: 실측상 val 이 더 나빠져 끔. 다시 시도하려면 True.
HFLIP = False

# 회피 oversampling: 차 감지+측면이동 큰 train 프레임을 N배 복제(재수집 없이).
# 회피 미세조정이 목적이면 켜둔 채(3) 새 회피 데이터를 무겁게 학습. (1=끔)
AVOID_OVERSAMPLE = 3

USE_VIZ = True
VIZ_EVERY = 5

cache_args = " ".join(f"--cache {shlex.quote(c)}" for c in CACHES)
resume_arg = f"--resume {shlex.quote(RESUME)}" if RESUME else ""
finetune_arg = "--finetune" if (RESUME and FINETUNE) else ""
wpfix_arg = "--wp_fix_sign" if WP_FIX_SIGN else ""
hflip_arg = "--hflip" if HFLIP else ""

viz_arg = ""
if USE_VIZ:
    viz_arg = f"--viz_dir {shlex.quote(VIZ_DIR)} --viz_every {VIZ_EVERY}"

cmd = (
    f"{sys.executable} train_e2e.py {cache_args} "
    f"--out {shlex.quote(CKPT)} "
    f"{viz_arg} "
    f"{resume_arg} "
    f"{finetune_arg} "
    f"--epochs {EPOCHS} "
    f"--batch {BATCH} "
    f"--lr {LR} "
    f"--val_frac {VAL_FRAC} "
    f"--workers {WORKERS} "
    f"--patience {PATIENCE} "
    f"--steer_weight {STEER_WEIGHT} "
    f"--throttle_weight {THROTTLE_WEIGHT} "
    f"--waypoint_weight {WAYPOINT_WEIGHT} "
    f"--steer_smooth {STEER_SMOOTH} "
    f"--avoid_oversample {AVOID_OVERSAMPLE} "
    f"{wpfix_arg} "
    f"{hflip_arg} "
    f"--device cuda"
)

print(cmd)
!{cmd}

In [ ]:
import glob
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

pngs = sorted(glob.glob(f"{VIZ_DIR}/*.png"))

last = []
if pngs:
    last_ep = max(os.path.basename(p).split("_")[0] for p in pngs)
    last = [p for p in pngs if os.path.basename(p).startswith(last_ep + "_")]

print(f"{len(pngs)} viz panels in {VIZ_DIR}; showing last epoch ({len(last)})")

n = min(6, len(last))

if n:
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(12, 3 * rows))
    axes = np.ravel(axes)

    for ax, p in zip(axes, last[:n]):
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(os.path.basename(p), fontsize=8)

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("no viz panels found")

In [ ]:
import sys

ONNX = "/kaggle/working/e2e.onnx"

!{sys.executable} export_onnx.py --ckpt {CKPT} --out {ONNX} --check

print("saved checkpoint:", CKPT)
print("saved onnx:", ONNX)